# Voz do Jarvis — treino no Colab

Ajusta a voz **Faber (pt_BR, Piper)** com o áudio gerado pela voz "Jarvis (UCM)". O resultado é um
`.onnx` que roda **offline** no Jarvis, tão rápido quanto a voz de hoje.

**Antes de começar**
1. No Colab: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
2. No seu Google Drive, crie a pasta `jarvis-voz` e suba nela o `jarvis-dataset.zip`.
3. Rode as células **em ordem**, uma por vez.

**Se a sessão cair:** rode tudo de novo. O treino continua do último ponto salvo no Drive
(`jarvis-voz/treino/last.ckpt`, copiado a cada 20 minutos).

Uso pessoal: a voz imita um dublador real. Não publique o modelo.

## 1. Configuração

In [ ]:
EPOCAS = 600          # ~2 h numa T4 com ~450 frases. Dá para parar antes e ouvir (célula 6)
LOTE = 16             # batch size; abaixe para 8 se faltar memória da GPU
DRIVE = "/content/drive/MyDrive/jarvis-voz"
NOME = "pt_BR-jarvis-medium"

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs(f"{DRIVE}/treino", exist_ok=True)
assert os.path.exists(f"{DRIVE}/jarvis-dataset.zip"), "Suba o jarvis-dataset.zip na pasta jarvis-voz do Drive"
print("ok:", os.listdir(DRIVE))

## 3. Instalar o Piper (treino) — uns 5 a 10 minutos

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
![ -d /content/piper1-gpl ] || git clone -q https://github.com/OHF-Voice/piper1-gpl /content/piper1-gpl
%cd /content/piper1-gpl
!pip -q install scikit-build onnx onnxscript  # onnxscript: o export do PyTorch novo pede
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
# o exportador novo do PyTorch (dynamo) não converte o preditor de duração: usa o antigo
!sed -i 's/        verbose=False,/        verbose=False,\n        dynamo=False,/' src/piper/train/export_onnx.py
!python3 setup.py build_ext --inplace > /dev/null
!python3 -c "from piper import espeakbridge; print('espeak ok')"
!python3 -m piper.train fit --help | head -3
%cd /content

## 4. Ponto de partida (voz Faber) e o áudio do Jarvis

In [ ]:
FABER = "/content/faber.ckpt"
if not os.path.exists(FABER):
    !wget -q -O {FABER} "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/pt/pt_BR/faber/medium/epoch%3D6159-step%3D1230728.ckpt"
!rm -rf /content/dataset && mkdir -p /content/dataset && unzip -q {DRIVE}/jarvis-dataset.zip -d /content/dataset
print(len(os.listdir("/content/dataset/wavs")), "áudios;", os.path.getsize(FABER) // 10**6, "MB de checkpoint")

## 5. Treinar

Primeira vez: parte da Faber (`warmstart_ckpt`: copia os pesos e começa a contagem do zero).
Depois de uma queda: continua do `last.ckpt` do Drive. A saída mostra `loss_g` e `loss_d` por época.

In [ ]:
import glob, shutil, subprocess, threading, time

LOCAL = "/content/treino"
os.makedirs(LOCAL, exist_ok=True)
retomar = f"{DRIVE}/treino/last.ckpt"
args = ["python3", "-m", "piper.train", "fit",
        "--data.voice_name", "jarvis",
        "--data.csv_path", "/content/dataset/metadata.csv",
        "--data.audio_dir", "/content/dataset/wavs",
        "--model.sample_rate", "22050",
        "--data.espeak_voice", "pt-br",
        "--data.cache_dir", "/content/cache",
        "--data.config_path", f"{DRIVE}/{NOME}.onnx.json",
        "--data.batch_size", str(LOTE),
        "--trainer.max_epochs", str(EPOCAS),
        "--trainer.default_root_dir", LOCAL]
if os.path.exists(retomar):
    print("continuando de", retomar)
    args += ["--ckpt_path", retomar]
else:
    args += ["--model.warmstart_ckpt", FABER]

def ultimo():
    achados = sorted(glob.glob(f"{LOCAL}/**/last.ckpt", recursive=True), key=os.path.getmtime)
    return achados[-1] if achados else None

parar = threading.Event()
def guardar():  # copia o último ponto para o Drive a cada 20 min
    while not parar.wait(20 * 60):
        u = ultimo()
        if u:
            shutil.copy(u, retomar + ".tmp"); os.replace(retomar + ".tmp", retomar)
            print(time.strftime("%H:%M"), "salvo no Drive", flush=True)
threading.Thread(target=guardar, daemon=True).start()

proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content/piper1-gpl")
for linha in proc.stdout:
    if any(p in linha for p in ("Epoch", "loss", "Error", "error", "Traceback", "warmstart", "INFO")):
        print(linha.rstrip()[-200:], flush=True)
proc.wait()
parar.set()
u = ultimo()
if u:
    shutil.copy(u, retomar)
print("fim, código", proc.returncode)

In [ ]:
# Onde está o treino (rode com a célula 5 parada: o treino segue em segundo plano)
import torch
u = ultimo() or retomar
ck = torch.load(u, map_location="cpu", weights_only=False)
print("época", ck["epoch"], "de", EPOCAS, "| passo", ck["global_step"])
print("treino rodando" if "proc" in dir() and proc.poll() is None else "treino parado")

## 6. Ouvir

Pode rodar com o treino parado a qualquer momento (botão de parar na célula 5). Exporta o último
ponto para `.onnx` e fala algumas frases.

In [ ]:
from IPython.display import Audio, display
ck = ultimo() or retomar
!python3 -m piper.train.export_onnx --checkpoint "{ck}" --output-file "/content/{NOME}.onnx"
shutil.copy(f"{DRIVE}/{NOME}.onnx.json", f"/content/{NOME}.onnx.json")
FRASES = ["Bom dia, senhor. Todos os sistemas estão operacionais.",
          "A próxima reunião começa daqui a quinze minutos.",
          "Agora faz dezoito graus em Belo Horizonte, com céu limpo, e o dia promete ser tranquilo.",
          "Tocando Bohemian Rhapsody, do Queen."]
assert os.path.exists(f"/content/{NOME}.onnx"), "a exportação falhou: veja o erro acima"
for i, f in enumerate(FRASES):
    !cd /content/piper1-gpl && python3 -m piper -m "/content/{NOME}.onnx" -f "/content/ouvir{i}.wav" -- "{f}"
    display(Audio(filename=f"/content/ouvir{i}.wav"))

## 7. Guardar a voz no Drive (baixe os dois arquivos para o computador)

In [ ]:
shutil.copy(f"/content/{NOME}.onnx", f"{DRIVE}/{NOME}.onnx")
print("No Drive, pasta jarvis-voz:")
print(f"  {NOME}.onnx")
print(f"  {NOME}.onnx.json")